## 🤖 Spark, PySpark, and Spark SQL Programming Assignment

**due: March 16th, 2025 (11:59pm)** \
**name:** Sardor Hazratov

#### part I: 🏠 median rental data (warm-up) 

In [0]:
from pyspark.sql import SparkSession

In [0]:
# create a new spark session
spark = SparkSession.builder.appName("big-data-assignment-1").getOrCreate()
print("Spark session initialized")

Spark session initialized


In [0]:
# create an RDD out of the data listed below:

median_rent = [
    ["Bronx", "borough", 2195, 2200],
    ["Brooklyn", "borough", 2999, 2999],
    ["Manhattan", "borough", 4000, 4100],
    ["Queens", "borough", 2500, 2495],
    ["Staten Island", "borough", 1600, 1600]
]
rdd = spark.sparkContext.parallelize(median_rent)
print("median_rent RDD created")

median_rent RDD created


In [0]:
# display how many partitions are used in this RDD
# (using getNumPartitions function)
print(f"Number of partitions: {rdd.getNumPartitions()}")


Number of partitions: 8


In [0]:
# create a new DataFrame from the RDD, with the following column names:
# "areaName", "areaType", "2023-12", "2024-01" 
df = spark.createDataFrame(rdd, ["areaName", "areaType", "2023-12", "2024-01"])
print("Created DF from RDD")

Created DF from RDD


In [0]:
# display the full 5 rows of the DataFrame
df.show(5, truncate=False)


+-------------+--------+-------+-------+
|areaName     |areaType|2023-12|2024-01|
+-------------+--------+-------+-------+
|Bronx        |borough |2195   |2200   |
|Brooklyn     |borough |2999   |2999   |
|Manhattan    |borough |4000   |4100   |
|Queens       |borough |2500   |2495   |
|Staten Island|borough |1600   |1600   |
+-------------+--------+-------+-------+



In [0]:
# create a new DataFrame that drops the "2023-12" column
# and uses DataFrame methods (not SQL) to sort the "2024-01" column, highest to lowest 
# display the full new DataFrame
new_df = df.drop("2023-12")
new_df = new_df.orderBy("2024-01", ascending=False)
new_df.show(truncate=False)

+-------------+--------+-------+
|areaName     |areaType|2024-01|
+-------------+--------+-------+
|Manhattan    |borough |4100   |
|Brooklyn     |borough |2999   |
|Queens       |borough |2495   |
|Bronx        |borough |2200   |
|Staten Island|borough |1600   |
+-------------+--------+-------+



✍️ Double-click to answer here in full sentence format:\
Look at your line of code in the previous cell. Of the methods that you used: which were **narrow transformations,** which were **wide transformations,** and which were **actions**? When did the series of transformations get "triggered" into actually computing?

- `drop()` - is a narrow transformation as it does not require data shuffling accross nodes.
- `orderBy()` - is a wide transformation as it needs to get all values to sort them. This requires all data to be shuffled accross nodes. 
- `show()` - is an action. Previous calls of transformations are just evaluated into logical plan (lazy evaluation) but were not triggered until an action of `show()` was called.


In [0]:
# create a new DataFrame including a new column, "normalized"
# the value of "normalized" is the value of the "2024-01" column, divided by 1000
# display the full new DataFrame
df_with_normalized = new_df.withColumn("normalized", new_df["2024-01"] / 1000)
df_with_normalized.show(truncate=False)


+-------------+--------+-------+----------+
|areaName     |areaType|2024-01|normalized|
+-------------+--------+-------+----------+
|Manhattan    |borough |4100   |4.1       |
|Brooklyn     |borough |2999   |2.999     |
|Queens       |borough |2495   |2.495     |
|Bronx        |borough |2200   |2.2       |
|Staten Island|borough |1600   |1.6       |
+-------------+--------+-------+----------+



#### part II: 🌎 common crawl data

In [0]:
# first read the overview of the Common Crawl project's mission, here on their website: https://commoncrawl.org/mission 

In [0]:
# uncomment & run the following code, which "gets" (downloads) a subset of the most recent common crawl 
# this may take a minute!

!wget https://data.commoncrawl.org/cc-index/table/cc-main/warc/crawl=CC-MAIN-2023-50/subset=crawldiagnostics/part-00000-e565b809-b335-4c1d-90fd-54a9a2b7113d.c000.gz.parquet

--2025-08-19 19:51:30--  https://data.commoncrawl.org/cc-index/table/cc-main/warc/crawl=CC-MAIN-2023-50/subset=crawldiagnostics/part-00000-e565b809-b335-4c1d-90fd-54a9a2b7113d.c000.gz.parquet
Resolving data.commoncrawl.org (data.commoncrawl.org)... 3.163.24.48, 3.163.24.102, 3.163.24.58, ...
Connecting to data.commoncrawl.org (data.commoncrawl.org)|3.163.24.48|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 207127367 (198M) [application/octet-stream]
Saving to: ‘part-00000-e565b809-b335-4c1d-90fd-54a9a2b7113d.c000.gz.parquet’

part-00000-e565b809 100%[===================>] 197.53M  33.7MB/s    in 6.1s    

2025-08-19 19:51:37 (32.5 MB/s) - ‘part-00000-e565b809-b335-4c1d-90fd-54a9a2b7113d.c000.gz.parquet’ saved [207127367/207127367]



In [0]:
%sh
# uncomment & run the following code, which lists the local file system 
# to check and see if that Parquet file made it

ls `pwd`

azure
conf
eventlogs
hadoop_accessed_config.lst
logs
part-00000-e565b809-b335-4c1d-90fd-54a9a2b7113d.c000.gz.parquet
preload_class.lst


In [0]:
# uncomment & run the following code, which uploads your downloaded Parquet file to the Databricks Distributed File System (and your account)

dbutils.fs.cp('file:/databricks/driver/part-00000-e565b809-b335-4c1d-90fd-54a9a2b7113d.c000.gz.parquet', 'dbfs:/FileStore/tables')

# after, in the menu to the left, click Catalog -> DBFS, look for the FileStore/tables folder, and check to see if your file made it there

Out[12]: True

In [0]:
# create a DataFrame from this Parquet file, using the `read` function
# hint: your file path should be dbfs:/FileStore/tables/part-00000-e565b809-b335-4c1d-90fd-54a9a2b7113d.c000.gz.parquet
common_crawl_df = spark.read.parquet("dbfs:/FileStore/tables/part-00000-e565b809-b335-4c1d-90fd-54a9a2b7113d.c000.gz.parquet")
print("Read parquet file")


Read parquet file


In [0]:
# remember, every DataFrame is built "on top of" an RDD. 
# uncomment & run the following code to access the underlying RDD
# and display the number of partitions used
# (replace YOURDATAFRAME with the name of your DataFrame from the previous cell)

print(f"Number of partitions: {common_crawl_df.rdd.getNumPartitions()}")


Number of partitions: 2


In [0]:
# print the schema for the DataFrame created by the Parquet file
common_crawl_df.printSchema()



root
 |-- url_surtkey: string (nullable = true)
 |-- url: string (nullable = true)
 |-- url_host_name: string (nullable = true)
 |-- url_host_tld: string (nullable = true)
 |-- url_host_2nd_last_part: string (nullable = true)
 |-- url_host_3rd_last_part: string (nullable = true)
 |-- url_host_4th_last_part: string (nullable = true)
 |-- url_host_5th_last_part: string (nullable = true)
 |-- url_host_registry_suffix: string (nullable = true)
 |-- url_host_registered_domain: string (nullable = true)
 |-- url_host_private_suffix: string (nullable = true)
 |-- url_host_private_domain: string (nullable = true)
 |-- url_host_name_reversed: string (nullable = true)
 |-- url_protocol: string (nullable = true)
 |-- url_port: integer (nullable = true)
 |-- url_path: string (nullable = true)
 |-- url_query: string (nullable = true)
 |-- fetch_time: timestamp (nullable = true)
 |-- fetch_status: short (nullable = true)
 |-- fetch_redirect: string (nullable = true)
 |-- content_digest: string (nulla

In [0]:
# looks like a lot of columns! 
# you can read more about each column here: https://data.commoncrawl.org/cc-index/table/cc-main/index.html 

# now: use an action to count the number of "rows" or elements in this DataFrame
number_of_rows = common_crawl_df.count()
print(f"Number of rows: {number_of_rows}")


Number of rows: 2565517


In [0]:
# display the first row in the DataFrame only
# use the argument truncate=False to show all the information
common_crawl_df.show(1, truncate=False)


+----------------------------------------------------------------+---------------------------------------------------------------------------+-------------------------------+------------+-----------------------+----------------------+----------------------+----------------------+------------------------+---------------------------+-----------------------+---------------------------+-------------------------------+------------+--------+------------------------------------+---------+-------------------+------------+--------------+--------------------------------+-----------------+---------------------+---------------+-----------------+-----------------+-------------------------------------------------------------------------------------------------------------------------+------------------+------------------+----------------+
|url_surtkey                                                     |url                                                                        |url_host_name        

In [0]:
# create a new DataFrame, called cc_filtered
# that uses the previous DataFrame but displays only the following columns:
# "url"
# "url_protocol"
# "url_host_3rd_last_part"
# "url_host_registered_domain"
# "url_host_registry_suffix"
# "content_mime_type"

# and then display the first 10 rows of cc_filtered
# use the argument truncate=True to clean up the display of the info

columns_to_pick = [
    "url", "url_protocol", "url_host_3rd_last_part", "url_host_registered_domain", "url_host_registry_suffix", "content_mime_type"
]
cc_filtered = common_crawl_df[columns_to_pick]
cc_filtered.show(10, truncate=True)


+--------------------+------------+----------------------+--------------------------+------------------------+-----------------+
|                 url|url_protocol|url_host_3rd_last_part|url_host_registered_domain|url_host_registry_suffix|content_mime_type|
+--------------------+------------+----------------------+--------------------------+------------------------+-----------------+
|https://www.safeh...|       https|                   www|      safehavencounseli...|                     com|        text/html|
|https://fr.safeha...|       https|                    fr|      safehavencounseli...|                     com|              unk|
|https://fr.safeha...|       https|                    fr|      safehavencounseli...|                     com|              unk|
|https://fr.safeha...|       https|                    fr|      safehavencounseli...|                     com|              unk|
|https://fr.safeha...|       https|                    fr|      safehavencounseli...|            

In [0]:
# re-name the following columns in cc_filtered
# "url_host_3rd_last_part" = "subdomain"
# "url_host_registry_suffix" = "suffix"
# "url_host_registered_domain" = "domain_name")

# and then display the first 3 rows of cc_filtered (with the new column names)
column_names_mapping = {
    "url_host_3rd_last_part": "subdomain",
    "url_host_registry_suffix": "suffix",
    "url_host_registered_domain": "domain_name",
}

major_version, minor_version = spark.version.split('.')[:2]
spark_curr_version = float(f"{major_version}.{minor_version}")
if spark_curr_version < 3.4:
    # legacy spark way of renaming column names
    for old_col_name, new_col_name in column_names_mapping.items():
        cc_filtered = cc_filtered.withColumnRenamed(old_col_name, new_col_name)
else:
    # withColumnsRenamed() added on 3.4.0
    # https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.withColumnsRenamed.html
    cc_filtered = cc_filtered.withColumnsRenamed(column_names_mapping)
cc_filtered.show(3, truncate=True)


+--------------------+------------+---------+--------------------+------+-----------------+
|                 url|url_protocol|subdomain|         domain_name|suffix|content_mime_type|
+--------------------+------------+---------+--------------------+------+-----------------+
|https://www.safeh...|       https|      www|safehavencounseli...|   com|        text/html|
|https://fr.safeha...|       https|       fr|safehavencounseli...|   com|              unk|
|https://fr.safeha...|       https|       fr|safehavencounseli...|   com|              unk|
+--------------------+------------+---------+--------------------+------+-----------------+
only showing top 3 rows



In [0]:
# we are going to use cc_filtered a lot - so let's cache it.
# write a line of code that caches cc_filtered
cc_filtered.cache()
_ = cc_filtered.collect()  # call some transformation to cache it
print("Dataframe cached")

Dataframe cached


In [0]:
# uncomment & run the following code to check: is the DataFrame cached?

print(cc_filtered.is_cached)

True


✍️ Double-click to answer here in full sentence format:\
is caching an **action** or a **transformation** in Spark? 
And why would I want to cache the DataFrame?

It is a transformation and it has no effect until acion is called. Caching is recommended when the data is used frequently to improve performance and speed of spark operations.

Ref: https://medium.com/@charchitpatidar/how-cache-works-in-apache-spark-aea6eeb3fd03


In [0]:
# uncomment & run the following code:
cc_mime = cc_filtered.select("content_mime_type").distinct()

In [0]:
# hmm, what is mime type? read up a little here: 
# https://developer.mozilla.org/en-US/docs/Web/HTTP/Basics_of_HTTP/MIME_types

# and show the first 60 rows of the DataFrame cc_mime
# include the argument truncate=False to show all the info
cc_mime.show(60, truncate=False)

+-----------------------------------------------------------------------+
|content_mime_type                                                      |
+-----------------------------------------------------------------------+
|text/x-perl                                                            |
|application/rss+xml                                                    |
|httpd/unix-directory                                                   |
|application/octet-stream                                               |
|redirect                                                               |
|application/x-httpd-ea-php54                                           |
|application/atom+xml                                                   |
|text/xml                                                               |
|text/Calendar                                                          |
|text/csv                                                               |
|application/xml                      

✍️ Double-click to answer here in full sentence format: \
what does the code in cell 27 (`select("content_mime_type").distinct()`) do? 
 
It selects disctinct values from "content_mime_type" column, that means only unique values are aggregated.  
_Note: this call is a transformation, we still did not call any action, like show() method._
  
 

In [0]:
# create a new DataFrame that counts up the total number of times each mime type occurs in the pages of this dataset
# and then filter so that the DataFrame only includes those types with more than 20 occurrences (or counts)
# and name the column with the total counts for each type "total_counts"
# and display the full results in descending order 

# use DataFrame methods (not SQL yet) to approach this
df_mime_type_counts = cc_filtered.groupBy("content_mime_type").count()
df_mime_type_counts = df_mime_type_counts.filter(df_mime_type_counts["count"] > 20)
df_mime_type_counts = df_mime_type_counts.withColumnRenamed("count", "total_counts")
df_mime_type_counts = df_mime_type_counts.sort("total_counts", ascending=False)
df_mime_type_counts.show(truncate=False)


+-----------------------------------------------------------------------+------------+
|content_mime_type                                                      |total_counts|
+-----------------------------------------------------------------------+------------+
|text/html                                                              |2206198     |
|unk                                                                    |273188      |
|warc/revisit                                                           |61926       |
|text/plain                                                             |19173       |
|application/json                                                       |1688        |
|application/octet-stream                                               |1240        |
|application/pdf                                                        |460         |
|application/binary                                                     |438         |
|application/rss+xml                       

In [0]:
# what are domains that show up the most in this dataset?
# create a new DataFrame that counts up the occurrences of each domain (grouping by the column "domain_name")
# show the first 50 rows of new DataFrame, including the argument truncate=False
df_domain_counts = cc_filtered.groupBy("domain_name").count()
df_domain_counts = df_domain_counts.sort("count", ascending=False)
df_domain_counts.show(50, truncate=False)



+------------------------+------+
|domain_name             |count |
+------------------------+------+
|sanook.com              |145516|
|salesforce.com          |54217 |
|saltwire.com            |39562 |
|sat24.com               |32986 |
|sap.com                 |22076 |
|sandro-paris.com        |19740 |
|samsung.com             |19445 |
|savedelete.com          |19195 |
|samsclub.com            |18891 |
|sams-sigma.com          |16478 |
|savvas.com              |12408 |
|sandiegouniontribune.com|12282 |
|sagepub.com             |11469 |
|salempress.com          |11099 |
|sas.com                 |8886  |
|sakuradk2.com           |7860  |
|sanitas.com             |7474  |
|saracens.com            |6983  |
|saint-gobain.com        |6841  |
|sanantoniomag.com       |5548  |
|sayweee.com             |4933  |
|saintpetersblog.com     |4702  |
|samsenalumni.com        |4512  |
|santiagoturismo.com     |4357  |
|sandzak.com             |4113  |
|savemate.com            |4078  |
|saglikeczane.

✍️ Look at your results from the previous code (cell 31). \
What does this tell us about the organization of this entire dataset? \
What do you think each line in `cc_filtered` represents?

The dataset contains information about crawled websites. Each line represents unique URL that was crawled. Each website is associated with `domain_name` and also divided by `subdomain`.  
Websites usually have more than one pages, and adding to this multiple subdomains delivers the possibility of having a huge number of pages. In this example `sanook.com` have 145516 unique pages.  



In [0]:
# create a new DataFrame holding all rows where domain_name = "sat24.com"
# use DataFrame methods (no SQL yet) to do this
# give the DataFrame only 1 column: "url"

# display the first 100 rows of this DataFrame, using the argument truncate=False
df_sat24 = cc_filtered.filter(cc_filtered["domain_name"] == "sat24.com").select("url")
df_sat24.show(100, truncate=False)



+------------------------------------------------------------------+
|url                                                               |
+------------------------------------------------------------------+
|https://www.sat24.com/                                            |
|https://sat24.com/                                                |
|https://www.sat24.com/                                            |
|https://www.sat24.com/                                            |
|https://www.sat24.com/                                            |
|http://www.sat24.com/                                             |
|https://www.sat24.com/                                            |
|https://www.sat24.com/                                            |
|https://www.sat24.com/                                            |
|https://www.sat24.com/                                            |
|https://www.sat24.com/                                            |
|https://www.sat24.com/           

In [0]:
# are there more pages using http or https in this dataset?
# use DataFrame methods to create a new DataFrame that counts up and groups by the "url_protocol" column
df_url_proto_count = cc_filtered.groupBy("url_protocol").count()
df_url_proto_count = df_url_proto_count.sort("count", ascending=False)
df_url_proto_count.show(truncate=False)

# There are more `https` url protocols 


+------------+-------+
|url_protocol|count  |
+------------+-------+
|https       |2028788|
|http        |536729 |
+------------+-------+



In [0]:
# what is the most common suffix (.com, .org, etc.) seen in this segment of the dataset?
# use DataFrame methods to create a new DataFrame that counts up and groups by the "suffix" column
df_suffix_count = cc_filtered.groupBy("suffix").count()
df_suffix_count = df_suffix_count.sort("count", ascending=False)
df_suffix_count.show(truncate=False)
## entire dataset contains only .com TLD

+------+-------+
|suffix|count  |
+------+-------+
|com   |2565517|
+------+-------+



In [0]:
# ... and now for some SQL!
# use the "createOrReplaceTempView" function on cc_filtered
# to create a view called "common_index"

cc_filtered.createOrReplaceTempView('common_index')



In [0]:
# use SQL to answer this question:
# how many pages in this segment of the dataset include the word "sailing" in their domain name?
# hint: check out the SQL LIKE keyword: https://www.w3schools.com/sql/sql_like.asp

# show the first 50 rows of your new DataFrame
# and then print the count of rows in that new DataFrame

keyword = "sailing"
query = f"""SELECT * FROM common_index WHERE domain_name like '%{keyword}%'"""
df_sailing = spark.sql(query)
df_sailing.show(50, truncate=False)
print(f"Total rows containing '{keyword}': {df_sailing.count()}")


+--------------------------------------------------------------------------------------------------------------------------+------------+---------+----------------------------+------+-----------------+
|url                                                                                                                       |url_protocol|subdomain|domain_name                 |suffix|content_mime_type|
+--------------------------------------------------------------------------------------------------------------------------+------------+---------+----------------------------+------+-----------------+
|https://saffron-sailing.com/destinations/langkawi-to-koh-lipe/                                                            |https       |null     |saffron-sailing.com         |com   |text/html        |
|http://www.safran-sailingteam.com/le-bateau/figaro/le-projet-127/                                                         |http        |www      |safran-sailingteam.com      |com   |unk      

In [0]:
# we are going to use DataFrame methods to ask a similar question:
# how many pages in this segment of the dataset might be translated into French?
# we are going to assume the page might be translated if it includes "/fr/" in the url
# hint: .contains("/fr/")

# create a new DataFrame with this French filter
# and drop all columns except "url" 
# show the first 10 rows of your new DataFrame, using truncate=False
# finally, print the count of rows in that new DataFrame

search_q = "/fr/"
df_french = cc_filtered.filter(cc_filtered["url"].contains(search_q)).select("url")
df_french.show(10, truncate=False)

print(f"Total rows translated into French: {df_french.count()}")


+---------------------------------------------------------------------------------+
|url                                                                              |
+---------------------------------------------------------------------------------+
|https://www.safehost.com/fr/centre-d-hebergement/sh3                             |
|https://safelagoon.com/fr/                                                       |
|https://safelightberlin.com/fr/blogs/news/silberra-the-path-of-the-pan-perfection|
|https://safelightberlin.com/fr/collections/35mm-slr                              |
|https://safelightberlin.com/fr/collections/bf22                                  |
|https://safelightberlin.com/fr/collections/lomography                            |
|https://safelightberlin.com/fr/collections/point-and-shoot                       |
|https://safelightberlin.com/fr/collections/silbersalz35                          |
|https://safelightberlin.com/fr/pages/contact-new                           

In [0]:
# what is the average page count per domain in this list?
# use whichever methods you choose (SQL or DataFrame functions)
# to approach this

# show your result as a new DataFrame with a single column called "average"

from pyspark.sql.functions import avg

df_avg_page_count = cc_filtered.groupBy("domain_name").count().withColumnRenamed("count", "total_pages")
df_avg_page_count.show(10, truncate=False)

df_avg_page_count = df_avg_page_count.select(avg("total_pages").alias("average"))

df_avg_page_count.show()

+---------------------------+-----------+
|domain_name                |total_pages|
+---------------------------+-----------+
|safehavencounselingpllc.com|10         |
|safehost.com               |1          |
|safehs.com                 |1          |
|safekeysdrivingschool.com  |2          |
|safeporntube.com           |2          |
|safeshipmoving.com         |2          |
|safesnout.com              |1          |
|safesoundfamily.com        |12         |
|safetran-traffic.com       |1          |
|safetycasesymposium.com    |66         |
+---------------------------+-----------+
only showing top 10 rows

+------------------+
|           average|
+------------------+
|22.041660223035553|
+------------------+



In [0]:
# now let's pull in some more data!
# uncomment & run the following code:
!wget https://data.commoncrawl.org/crawl-data/CC-MAIN-2023-50/segments/1700679099281.67/wet/CC-MAIN-20231128083443-20231128113443-00000.warc.wet.gz



--2025-08-19 19:53:03--  https://data.commoncrawl.org/crawl-data/CC-MAIN-2023-50/segments/1700679099281.67/wet/CC-MAIN-20231128083443-20231128113443-00000.warc.wet.gz
Resolving data.commoncrawl.org (data.commoncrawl.org)... 18.161.6.121, 18.161.6.27, 18.161.6.34, ...
Connecting to data.commoncrawl.org (data.commoncrawl.org)|18.161.6.121|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 112806267 (108M) [application/octet-stream]
Saving to: ‘CC-MAIN-20231128083443-20231128113443-00000.warc.wet.gz’

CC-MAIN-20231128083 100%[===================>] 107.58M  42.7MB/s    in 2.5s    

2025-08-19 19:53:06 (42.7 MB/s) - ‘CC-MAIN-20231128083443-20231128113443-00000.warc.wet.gz’ saved [112806267/112806267]



In [0]:
# uncomment & run the following code:
!zcat CC-MAIN-20231128083443-20231128113443-00000.warc.wet.gz | head -n 500

# this reads the first 500 lines of the file you downloaded into the notebook's local file system (not distributed file)
































Новости 0-50.ru | Порошенко выразил соболезнования родным и близким Валерии Новодворской |
|
Погода в Екатеринбурге и Свердловской области |
Что приготовить на ужин рецепт с фото |
Новости Екатеринбурга |
96women.ru - Живой Женский Журнал |
Главная
Редакция
Реклама
О проекте
RSS
Обратная связь
Все новости
Екатеринбург
Россия и мир
Образование
Недвижимость
Здоровье
Спорт
Происшествия
Транспорт
Новости компаний
Другая жизнь
Статьи
Порошенко выразил соболезнования родным и близким Валерии Новодворской
Президент Украины Петр Порошенко выразил свои соболезнования родным и близким Валерии Новодворской, скончавшейся накануне.
"Патриотка России и друг Украины, она еще с момента вторжения советских войск в Чехословакию жила с лозунгом "За нашу и вашу свободу"! Выражаю грусть и глубокое соболезнование родным и близким покойной, всем тем, для кого Валерия Ильинична была авторитетным и дорогим человеком", – говорится в сообщении Порошенко в Facebook.
Правозащитница Нов

✍️  What kind of data is this, compared to the first dataset?

This is WET (Web Extracted Text) format dataset.  It contains simple plain text extracted from web content while crawling. It does not contain any html or links. This is useful for text analysis and NLP.  
Data is stored in one long file.  


Ref:  
https://commoncrawl.org/blog/web-archiving-file-formats-explained  


In [0]:
# let's pull in more common crawl data!
# uncomment & run the following code:
!wget https://data.commoncrawl.org/crawl-data/CC-MAIN-2023-50/segments/1700679099281.67/warc/CC-MAIN-20231128083443-20231128113443-00000.warc.gz


--2025-08-19 19:53:08--  https://data.commoncrawl.org/crawl-data/CC-MAIN-2023-50/segments/1700679099281.67/warc/CC-MAIN-20231128083443-20231128113443-00000.warc.gz
Resolving data.commoncrawl.org (data.commoncrawl.org)... 18.161.6.75, 18.161.6.34, 18.161.6.27, ...
Connecting to data.commoncrawl.org (data.commoncrawl.org)|18.161.6.75|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1208969113 (1.1G) [application/octet-stream]
Saving to: ‘CC-MAIN-20231128083443-20231128113443-00000.warc.gz’

CC-MAIN-20231128083 100%[===================>]   1.12G  56.6MB/s    in 21s     

2025-08-19 19:53:29 (56.2 MB/s) - ‘CC-MAIN-20231128083443-20231128113443-00000.warc.gz’ saved [1208969113/1208969113]



In [0]:
# uncomment & run the following code:
!zcat CC-MAIN-20231128083443-20231128113443-00000.warc.gz | head -n 500








































































<!doctype html>
<html lang="ru">
<head>
	<meta charset="windows-1251">
	<title>������� 0-50.ru | ��������� ������� �������������� ������ � ������� ������� ������������ | </title>
	<meta name="title" content="��������� ������� �������������� ������ � ������� ������� ������������">
	<meta property="og:title" content="��������� ������� �������������� ������ � ������� ������� ������������">
	<meta property="og:image" content="http://0-50.ru/images/foto/49761_full.jpg">
	<meta name="description" content="��������� ������� ���� ��������� ������� ���� �������������� ������ � ������� ������� ������������, ������������ ��������.">
	<meta name="Author" content="0-50.ru">
	<meta name="twitter:image:src" content="http://0-50.ru/images/foto/49761_full.jpg">
	
	<!--[if lt IE 9]>
		<script src="http://html5shim.googlecode.com/svn/trunk/html5.js"></script>
	<![endif]-->
	<meta name="rp51afcaa28a694b82948bd3a36a0490f8" content="f3aeec7

✍️  Now, what kind of data is **this**, compared to the first dataset and the second file?

This is Web Archive (WARC) format dataset. It contains a set of html text headers and raw web content from crawled websites.  
Data is stored in one long file.  


Ref:  
https://commoncrawl.org/blog/web-archiving-file-formats-explained  
https://iipc.github.io/warc-specifications/specifications/warc-format/warc-1.0/  

In [0]:
# uncomment & run the following code:
!gunzip CC-MAIN-20231128083443-20231128113443-00000.warc.gz

# this code "unzips" the compressed .gz file and may take a minute - !

In [0]:
# uncomment & run the following code 
# to upload the file to the DBFS distributed file system  - this might take a few minutes!

dbutils.fs.cp('file:/databricks/driver/CC-MAIN-20231128083443-20231128113443-00000.warc', 'dbfs:/FileStore/tables')

# wait until the Output below says "True," and you could also check for the file in the Catalog -> DBFS menu on the left.

Out[38]: True

In [0]:
# create an RDD with this most recent data using the .textFile function
# call this RDD warc_rdd
# hint: the file path should be dbfs:/FileStore/tables/CC-MAIN-20231128083443-20231128113443-00000.warc

warc_rdd = spark.sparkContext.textFile("dbfs:/FileStore/tables/CC-MAIN-20231128083443-20231128113443-00000.warc")

In [0]:
# how many partitions are used in this RDD?
print(f"Number of partitions: {warc_rdd.getNumPartitions()}")


Number of partitions: 84


In [0]:
# display the first 500 elements in this RDD

warc_rdd.take(500)

Out[41]: ['WARC/1.0',
 'WARC-Type: warcinfo',
 'WARC-Date: 2023-11-28T08:34:43Z',
 'WARC-Record-ID: <urn:uuid:2b6daf87-e3fb-4851-a842-b24612e49256>',
 'Content-Length: 499',
 'Content-Type: application/warc-fields',
 'WARC-Filename: CC-MAIN-20231128083443-20231128113443-00000.warc.gz',
 '',
 'isPartOf: CC-MAIN-2023-50',
 'publisher: Common Crawl',
 'description: Wide crawl of the web for November/December 2023',
 'operator: Common Crawl Admin (info@commoncrawl.org)',
 'hostname: ip-10-67-67-56',
 'software: Apache Nutch 1.19 (modified, https://github.com/commoncrawl/nutch/)',
 'robots: checked via crawler-commons 1.5-SNAPSHOT (https://github.com/crawler-commons/crawler-commons)',
 'format: WARC File Format 1.1',
 'conformsTo: https://iipc.github.io/warc-specifications/specifications/warc-format/warc-1.1/',
 '',
 '',
 'WARC/1.0',
 'WARC-Type: request',
 'WARC-Date: 2023-11-28T11:34:01Z',
 'WARC-Record-ID: <urn:uuid:0ca80b2d-5a43-41a0-8809-bfe4626f2f8d>',
 'Content-Length: 277',
 'Conten

✍️  What does each "element" of the RDD here represent?

Each element represents a line in the file. This data is unstructured compared to parquet file which is structured.

In [0]:
# you are given the function below:

def count_script_tags(line):
    return line.lower().count('<script type="text/javascript">')

# use this function, and lambda functions, to count the number of times that Javascript tag
# appears in this segment of the dataset
# hint: you will want to use a mapping function, and `.reduce()` 

# print the resulting value, the final count

warc_rdd.map(lambda x: count_script_tags(x)).reduce(lambda x,y: x+y)

Out[42]: 91982

In [0]:
# there are usually multiple ways of accomplishing the same task ...
# use an accumulator variable to count the number of Javascript tags 
# instead of the mapping + reducing that you did previously

# print the resulting value of the accumulator variable

js_tags = spark.sparkContext.accumulator(0)

warc_rdd.foreach(lambda l: js_tags.add(count_script_tags(l)))

print(js_tags)

91982


✍️  Check out the Spark UI (under View menu) to investigate whether there was any performance difference between \
your 2 approaches for counting the Javascript tags.

![](https://i.imgur.com/8sRJzXT.png)

I dont see significant difference on each method. Though I observed that computation time varies on each time.  
For map-reduce method it took 2.9 nd 2.6 minutes, while for aggregator func it took 2.7 and 2.7 minutes on each observance.  


In [0]:
# let's try and figure out the average content length of these pages from the common crawl
# each page gives that information in a header in the data
# first: create a new RDD that filters warc_rdd
# using a lambda function that runs on each line: 
# if the string 'Content-Length' from the header is in the line, then that row should be in the new RDD

# display the first 10 rows of this RDD




# I am using case-insensitive. maybe there is some HTTP/1.1 and HTTP/2 difference
# HTTP/2 has lowercase enforcement
warc_rdd_content_len = warc_rdd.filter(lambda x: "content-length" in x.lower())
warc_rdd_content_len.take(10)


Out[44]: ['Content-Length: 499',
 'Content-Length: 277',
 'Content-Length: 42587',
 'Content-Length: 42070',
 'Content-Length: 208',
 'Content-Length: 327',
 'Content-Length: 39949',
 'Content-Length: 39528',
 'Content-Length: 202',
 'Content-Length: 295']

In [0]:
# the goal now = to get a DataFrame with only the content length, as a number, on each line

# you are given the following function, which parses the text on each line and returns tuples:
# (the function uses regular expressions, or RegEx, to search through strings for numbers)
# (more on RegEx: https://en.wikipedia.org/wiki/Regular_expression)

import re
# import the re (or RegEx) library

def parse_length_line(line):
    # regular expression to find 'Content-Length:' followed by any number of digits
    match = re.search(r'Content-Length:\s*(\d+)', line)
    if match:
        # if a match is found, convert the matching group (the digits) to an integer
        return (int(match.group(1)),)
    else:
        # if no match, return None
        return (None,)
    
# using RDD methods, run this function on every line in the RDD you just created in the previous cell
# and then convert the result to a new DataFrame called length_df
# it should have only 1 column: "Length"

warc_rdd_content_len_int = warc_rdd_content_len.map(lambda l: parse_length_line(l))
length_df = warc_rdd_content_len_int.toDF(["Length"])

In [0]:
# show the first 10 rows of length_df
# and print the schema of this DataFrame

length_df.show(10)
length_df.printSchema()

+------+
|Length|
+------+
|   499|
|   277|
| 42587|
| 42070|
|   208|
|   327|
| 39949|
| 39528|
|   202|
|   295|
+------+
only showing top 10 rows

root
 |-- Length: long (nullable = true)



In [0]:
# now for some SQL!
# use createOrReplaceTempView to create a view based on length_df
# called "content_length"

length_df.createOrReplaceTempView('content_length')


In [0]:
# use SQL to print out the average content length for pages in this segment of the dataset
query = """SELECT AVG(*) as AverageContentLength FROM content_length"""
spark.sql(query).show()



+--------------------+
|AverageContentLength|
+--------------------+
|   93421.36533395662|
+--------------------+



In [0]:
# invent your own question to query from this dataset, or previous DataFrames in this notebook
# you may use any methods that you choose to answer the question

# how many html documents annotated its language by lang="" attribute. classify unannotated ones as "unknown"
# For example documents crawled:
# <html lang="ja"
# <html lang="en-US"
# <html lang="ru" 
# <html lang="pl-PL"
# <html >                                           <- !! unknown
# <html xmlns="http://www.w3.org/1999/xhtml">       <- !! unknown

html_tags_rdd = warc_rdd.filter(lambda x: "<html" in x.lower())  # only lines with <html

def extract_lang(line: str) -> str:
    regex_lang = re.compile(r'<html lang=\"([-\w]+)\"')  # matches "en" "en-US" or others similar
    matched = regex_lang.search(line)
    if matched:
        return matched.group(1)

    return "unknown"

html_lang_rdd = html_tags_rdd.map(lambda l: extract_lang(l))
# output
# ['ru','en-US','tr-TR','unknown','unknown','pl-PL','en-US','unknown','en-US','es-AR']

# now I am extracting only first 2 letters.
# I think that en-US, en-GB, en, en_US all can be categorized as simple "en"
# unless it is unknown
def clean_lang(l):
    if l == "unknown":
        return l, 1 # no change
    return l[:2], 1
html_lang_rdd = html_lang_rdd.map(lambda l: clean_lang(l))
# [('ru', 1), ('en', 1), ('tr', 1), ('unknown', 1), ('unknown', 1), ('pl', 1), ('en', 1), ('unknown', 1), ('en', 1), ('es', 1)]
html_lang_rdd = html_lang_rdd.reduceByKey(lambda x,y: x+y).sortBy(lambda x: x[1], ascending=False)
df_langs = html_lang_rdd.toDF(["lang", "count"])
df_langs.show(100)  # top 100

+-------+-----+
|   lang|count|
+-------+-----+
|unknown|23202|
|     en|10730|
|     de| 1282|
|     ru| 1168|
|     fr| 1155|
|     es| 1146|
|     ja| 1138|
|     it|  655|
|     zh|  549|
|     nl|  520|
|     pl|  475|
|     pt|  474|
|     vi|  399|
|     tr|  240|
|     cs|  221|
|     ko|  174|
|     sv|  169|
|     ro|  162|
|     hu|  129|
|     id|  124|
|     da|  107|
|     el|  102|
|     uk|   98|
|     fi|   87|
|     sk|   85|
|     th|   83|
|     bg|   65|
|     nb|   52|
|     sr|   51|
|     fa|   49|
|     ar|   43|
|     sl|   43|
|     hr|   43|
|     ca|   38|
|     lt|   35|
|     et|   34|
|     lv|   29|
|     he|   28|
|     no|   27|
|     az|   14|
|     jp|   12|
|     ua|   12|
|     eu|   11|
|     ka|   10|
|     bs|    9|
|     zx|    8|
|     hi|    7|
|     us|    7|
|     ms|    6|
|     sq|    6|
|     EN|    6|
|     is|    6|
|     cn|    5|
|     gr|    5|
|     eo|    5|
|     gl|    5|
|     cy|    5|
|     nn|    5|
|     mk|    5|
|     cz

✍️  Please **cite any sources** that you used, other than class notes and the Codecademy course, \
to help with or complete this notebook. You should list any websites, tools, videos, etc.!

- https://commoncrawl.org/blog/web-archiving-file-formats-explained  
- https://iipc.github.io/warc-specifications/specifications/warc-format/warc-1.0/  
- https://stackoverflow.com/  
- https://datascience.stackexchange.com/  
- https://spark.apache.org/docs/3.5.4/  